## When mergers are found

We consider whern mergers would be found in the data by the two searches and produce Table X - the table of times-before-merger that a signal is found

In [1]:
import numpy as np

import sys
import os

parent_dir = os.path.abspath("..")

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from common_utils import get_results_zero_latency, get_results_inpainting

/home/gareth.cabourndavies/environments/env_lisa_premerger_sangria/lib/python3.11/site-packages/pycbc/types/array.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal as _lal


In [2]:
times_before = [14,7,4,1,0.5]
results_zero_latency = get_results_zero_latency(
    "zero_latency/results/psd_estimate/data_runs_remove_{time_before}_results.hdf",
    times_before
)

results_inpainting = get_results_inpainting(
    "inpainting_runs/results/data_runs_psd_estimate.hdf"
)

In [3]:
# Get the truth times
import ldc.io.hdf5 as hdfio

input_data = '../datasets/LDC2_sangria_hm_training.hdf'
mbhb_sky, _ = hdfio.load_array(input_data, name="sky/mbhb/cat")

truth_times_s = mbhb_sky['CoalescenceTime']

In [4]:
times_before_ip = np.unique(results_inpainting['time_before'])[::-1]
t_window = 7200
zl_snr_limit = 10
ip_snr_limit = 10
table_str = """\\begin{tabular}{|c|cc|cc|c|}
\\hline
\\multirow{3}{*}{Signal Number} & \\multicolumn{4}{c|}{Found} & \\multirow{3}{*}{Extra Warning (days)} \\\\
\\cline{2-5}
& \\multicolumn{2}{c|}{Zero Latency} & \\multicolumn{2}{c|}{Inpainting} & \\\\
\\cline{2-5}
 & Time (days)  & SNR & Time (days)  & SNR & \\\\
\hline
"""
for signal_number in range(15):
    # Signals in the congested region get skipped
    if signal_number in [2,3,4,5]:
        continue
    zl_time_before = None
    zl_tbefore_str = '\multicolumn{2}{c|}{Never}'
    zl_snr = ' '

    for t_before in times_before:
        this_t = results_zero_latency['time_before'] == t_before
        within_window = abs(results_zero_latency['time'][this_t] - truth_times_s[signal_number]) < t_window
        if results_zero_latency['snr'][this_t][within_window].max() > zl_snr_limit:
            # We have found something - update the result strings
            which_result = results_zero_latency['snr'][this_t][within_window].argmax()
            zl_snr = '%.3f' % results_zero_latency['snr'][this_t][within_window][which_result]
            zl_time_before = t_before
            zl_tbefore_str = f'{t_before} & '
            # We are not bothered about the signal being found after the original result, so skip it
            break


    # if zl_snr != ' ':
    #     print(f'Zero Latency: Signal {signal_number} found {zl_time_before} days before merger with SNR {zl_snr}')

    ip_time_before = None
    ip_tbefore_str = '\multicolumn{2}{c|}{Never}'
    ip_snr = ' '

    for t_before in times_before_ip:
        this_t = results_inpainting['time_before'] == t_before
        within_window = abs(results_inpainting['time'][this_t] - truth_times_s[signal_number]) < t_window
        if results_inpainting['snr'][this_t][within_window].max() > ip_snr_limit:
            # We have found something - update the result strings
            which_result = results_inpainting['snr'][this_t][within_window].argmax()
            ip_snr = '%.3f' % results_inpainting['snr'][this_t][within_window][which_result]
            ip_time_before = t_before
            ip_tbefore_str = f'{t_before:.2f} & '
            # We are not bothered about the signal being found after the original result, so skip it
            break

    if ip_time_before is None:
        extra_warning = 'N/A'
        ip_snr = ' '
    elif zl_time_before is None:
        extra_warning = f'{ip_time_before:.2f}'
        zl_snr = ' '
    else:
        extra_warning = f'{ip_time_before - zl_time_before:.2f}'


    # if ip_snr_found != ' ':
    #     print(f'Inpainting: Signal {signal_number} found {ip_time_before} days before merger with SNR {ip_snr}')

    table_line =  f"{signal_number} & {zl_tbefore_str}  {zl_snr} & {ip_tbefore_str} {ip_snr} & {extra_warning} \\\\\n"
    
    table_str += table_line

table_end = """\\hline
\\end{tabular}"""
table_str += table_end

print(table_str)
with open('../paper/tables/results_when_found.tex', 'w') as result_f:
    result_f.write(table_str)

\begin{tabular}{|c|cc|cc|c|}
\hline
\multirow{3}{*}{Signal Number} & \multicolumn{4}{c|}{Found} & \multirow{3}{*}{Extra Warning (days)} \\
\cline{2-5}
& \multicolumn{2}{c|}{Zero Latency} & \multicolumn{2}{c|}{Inpainting} & \\
\cline{2-5}
 & Time (days)  & SNR & Time (days)  & SNR & \\
\hline
0 & 7 &   14.665 & 14.00 &  10.632 & 7.00 \\
1 & \multicolumn{2}{c|}{Never}    & \multicolumn{2}{c|}{Never}   & N/A \\
6 & 1 &   14.343 & 1.67 &  10.191 & 0.67 \\
7 & 0.5 &   15.838 & 0.96 &  10.346 & 0.46 \\
8 & 1 &   12.596 & 1.50 &  10.151 & 0.50 \\
9 & 1 &   12.051 & 1.33 &  10.101 & 0.33 \\
10 & 4 &   11.870 & 4.96 &  10.188 & 0.96 \\
11 & 4 &   13.896 & 6.71 &  10.071 & 2.71 \\
12 & 1 &   12.188 & 1.58 &  10.241 & 0.58 \\
13 & 1 &   17.453 & 2.67 &  10.118 & 1.67 \\
14 & 0.5 &   11.177 & 0.54 &  10.590 & 0.04 \\
\hline
\end{tabular}
